# 05. SAS-Python Cross-Platform Validation


Short summary:
[05 - SAS-Python Reproducibility Validation](../docs/05_sas_python_validation_SHORT.md)

## 1. Purpose

This notebook is a lightweight validation layer between independently generated Python and SAS outputs for the non-ICU RAAS project.

It compares exported CSV artifacts. It does not perform new statistical modeling, rebuild upstream summaries, or reinterpret the clinical analysis.


## 2. Scope

The notebook is limited to four tasks:

1. Load Python output CSV files.
2. Load SAS output CSV files.
3. Apply minimal schema alignment where file naming conventions differ.
4. Compare values and summarize discrepancies.

The statistical analysis logic belongs upstream in notebooks `01`-`04b` and SAS programs `01`-`04`. This notebook should remain an output comparison layer only.


## 3. Methods

Validation is organized by analysis stage:

| Stage | Python source | Python output | SAS source | SAS output |
|---|---|---|---|---|
| 04a unadjusted outcomes | `notebooks/04a_unadjusted_outcomes.ipynb` | `python/outputs/python_unadjusted_outcomes.csv` | `sas/programs/03_unadjusted_analysis.sas` | `sas/outputs/sas_unadjusted_outcomes.csv` |
| 04b multivariable model | `notebooks/04b_multivariable_outcomes.ipynb` | `python/outputs/python_logistic_parameters.csv` | `sas/programs/04_multivariable_logistic.sas` | `sas/outputs/sas_logistic_parameters.csv` |

Minimal normalization is acceptable when it makes independently generated outputs comparable, such as column renaming, type coercion, sorting before merge, and explicit term-name aliasing. Hidden fallback calculations and reconstructed model outputs are intentionally avoided.


## 4. Quality Checks

Flagged differences should be reviewed in context. Differences can arise from categorical reference levels, dummy-variable coding, numeric precision, rounding, convergence behavior, or quasi-complete separation in sparse categories.

The multivariable model includes `admission_type` categories with extreme odds ratios. These terms are especially sensitive to sparse cells and rare outcomes, so implementation-specific differences may appear even when the model specification is unchanged.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


In [2]:
NOTEBOOKS_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
PROJECT_ROOT = NOTEBOOKS_DIR.parent

PYTHON_UNADJUSTED_PATH = PROJECT_ROOT / "python" / "outputs" / "python_unadjusted_outcomes.csv"
SAS_UNADJUSTED_PATH = PROJECT_ROOT / "sas" / "outputs" / "sas_unadjusted_outcomes.csv"
PYTHON_LOGISTIC_PATH = PROJECT_ROOT / "python" / "outputs" / "python_logistic_parameters.csv"
SAS_LOGISTIC_PATH = PROJECT_ROOT / "sas" / "outputs" / "sas_logistic_parameters.csv"

expected_files = pd.DataFrame(
    [
        ("04a", "Python", PYTHON_UNADJUSTED_PATH),
        ("04a", "SAS", SAS_UNADJUSTED_PATH),
        ("04b", "Python", PYTHON_LOGISTIC_PATH),
        ("04b", "SAS", SAS_LOGISTIC_PATH),
    ],
    columns=["stage", "platform", "path"],
)
expected_files["relative_path"] = expected_files["path"].map(lambda p: str(p.relative_to(PROJECT_ROOT)))
expected_files["available"] = expected_files["path"].map(Path.exists)
expected_files[["stage", "platform", "relative_path", "available"]]


,stage,platform,relative_path,available
0,04a,Python,python/outputs/python_unadjusted_outcomes.csv,True
1,04a,SAS,sas/outputs/sas_unadjusted_outcomes.csv,True
2,04b,Python,python/outputs/python_logistic_parameters.csv,True
3,04b,SAS,sas/outputs/sas_logistic_parameters.csv,True


In [3]:
def load_csv(path: Path, label: str, required: bool = False) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)

    message = f"TODO: `{path.relative_to(PROJECT_ROOT)}` is missing. Generate the {label} output before running validation."
    if required:
        raise FileNotFoundError(message)
    display(Markdown(message))
    return None


python_unadjusted_raw = load_csv(PYTHON_UNADJUSTED_PATH, "Python unadjusted", required=True)
sas_unadjusted_raw = load_csv(SAS_UNADJUSTED_PATH, "SAS unadjusted")
python_logistic_raw = load_csv(PYTHON_LOGISTIC_PATH, "Python logistic", required=True)
sas_logistic_raw = load_csv(SAS_LOGISTIC_PATH, "SAS logistic")


def dataframe_overview(label: str, df: pd.DataFrame | None) -> dict:
    if df is None:
        return {"dataset": label, "rows": 0, "columns": 0, "available": False}
    return {"dataset": label, "rows": len(df), "columns": len(df.columns), "available": True}


def comparison_overview(stage: str, comparison: pd.DataFrame) -> pd.DataFrame:
    if comparison.empty:
        return pd.DataFrame(
            [{"stage": stage, "comparison_rows": 0, "passed_rows": 0, "failed_rows": 0, "max_abs_difference": np.nan}]
        )

    return pd.DataFrame(
        [
            {
                "stage": stage,
                "comparison_rows": len(comparison),
                "passed_rows": int(comparison["pass"].sum()),
                "failed_rows": int((~comparison["pass"]).sum()),
                "max_abs_difference": comparison["absolute_difference"].replace([np.inf, -np.inf], np.nan).max(),
            }
        ]
    )


def discrepancy_overview(discrepancies: pd.DataFrame, group_column: str) -> pd.DataFrame:
    if discrepancies.empty:
        return pd.DataFrame(columns=[group_column, "failed_rows", "max_abs_difference", "max_relative_difference"])

    return (
        discrepancies
        .groupby(group_column, as_index=False)
        .agg(
            failed_rows=(group_column, "size"),
            max_abs_difference=("absolute_difference", "max"),
            max_relative_difference=("relative_difference", "max"),
        )
        .sort_values(["failed_rows", group_column], ascending=[False, True])
        .reset_index(drop=True)
    )

## 5. Validation 1: 04a Unadjusted Outcomes

**Python source**: `notebooks/04a_unadjusted_outcomes.ipynb`  
**SAS source**: `sas/programs/03_unadjusted_analysis.sas`  
**Files compared**: `python/outputs/python_unadjusted_outcomes.csv` vs `sas/outputs/sas_unadjusted_outcomes.csv`

Variables compared:

- `exposure`
- `exposure_group`
- `n`
- `deaths`
- `non_deaths`
- `mortality_proportion`
- `crude_odds`
- `crude_odds_ratio_vs_no_early_raas`

Tolerance: numeric absolute difference <= `1e-8`. String fields must match exactly after string coercion.


In [4]:
UNADJUSTED_COLUMN_RENAMES = {
    "crude_or_vs_no_raas": "crude_odds_ratio_vs_no_early_raas",
}

UNADJUSTED_KEY = ["exposure"]
UNADJUSTED_COMPARE_COLUMNS = [
    "exposure_group",
    "n",
    "deaths",
    "non_deaths",
    "mortality_proportion",
    "crude_odds",
    "crude_odds_ratio_vs_no_early_raas",
]
UNADJUSTED_NUMERIC_COLUMNS = [
    "n",
    "deaths",
    "non_deaths",
    "mortality_proportion",
    "crude_odds",
    "crude_odds_ratio_vs_no_early_raas",
]
UNADJUSTED_ABS_TOL = 1e-8


def prepare_unadjusted(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    out = df.rename(columns=UNADJUSTED_COLUMN_RENAMES).copy()
    expected = UNADJUSTED_KEY + UNADJUSTED_COMPARE_COLUMNS
    missing = [col for col in expected if col not in out.columns]
    if missing:
        raise ValueError(f"{platform} unadjusted output is missing columns: {missing}")

    out = out[expected].copy()
    out["exposure"] = pd.to_numeric(out["exposure"], errors="coerce")
    for col in UNADJUSTED_NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.sort_values(UNADJUSTED_KEY).reset_index(drop=True)

python_unadjusted = prepare_unadjusted(python_unadjusted_raw, "Python")
sas_unadjusted = None if sas_unadjusted_raw is None else prepare_unadjusted(sas_unadjusted_raw, "SAS")

pd.DataFrame(
    [
        dataframe_overview("Python unadjusted", python_unadjusted),
        dataframe_overview("SAS unadjusted", sas_unadjusted),
    ]
)

,dataset,rows,columns,available
0,Python unadjusted,2,8,True
1,SAS unadjusted,2,8,True


In [5]:
def compare_table(
    python_df: pd.DataFrame,
    sas_df: pd.DataFrame,
    key_columns: list[str],
    compare_columns: list[str],
    numeric_columns: list[str],
    abs_tolerance: float,
) -> pd.DataFrame:
    merged = python_df.merge(
        sas_df,
        on=key_columns,
        how="outer",
        suffixes=("_python", "_sas"),
        indicator=True,
    )

    rows = []
    for _, row in merged.iterrows():
        key_label = ", ".join(f"{key}={row[key]}" for key in key_columns)
        for variable in compare_columns:
            python_value = row.get(f"{variable}_python")
            sas_value = row.get(f"{variable}_sas")
            is_numeric = variable in numeric_columns

            if row["_merge"] != "both":
                absolute_difference = np.nan
                relative_difference = np.nan
                passed = False
            elif is_numeric:
                absolute_difference = abs(sas_value - python_value)
                denominator = max(abs(python_value), 1e-12)
                relative_difference = absolute_difference / denominator
                passed = bool(absolute_difference <= abs_tolerance)
            else:
                absolute_difference = np.nan
                relative_difference = np.nan
                passed = bool(str(python_value) == str(sas_value))

            rows.append(
                {
                    "key": key_label,
                    "variable": variable,
                    "python_value": python_value,
                    "sas_value": sas_value,
                    "absolute_difference": absolute_difference,
                    "relative_difference": relative_difference,
                    "pass": passed,
                }
            )
    return pd.DataFrame(rows)

if sas_unadjusted is None:
    display(Markdown("TODO: SAS unadjusted output is unavailable; comparison table was not generated."))
    unadjusted_comparison = pd.DataFrame()
else:
    unadjusted_comparison = compare_table(
        python_df=python_unadjusted,
        sas_df=sas_unadjusted,
        key_columns=UNADJUSTED_KEY,
        compare_columns=UNADJUSTED_COMPARE_COLUMNS,
        numeric_columns=UNADJUSTED_NUMERIC_COLUMNS,
        abs_tolerance=UNADJUSTED_ABS_TOL,
    )

comparison_overview("04a unadjusted outcomes", unadjusted_comparison)

,stage,comparison_rows,passed_rows,failed_rows,max_abs_difference
0,04a unadjusted outcomes,14,14,0,5.200285e-13


In [6]:
unadjusted_discrepancies = unadjusted_comparison.loc[
    unadjusted_comparison["pass"].eq(False)
].copy() if not unadjusted_comparison.empty else pd.DataFrame()

unadjusted_discrepancy_summary = discrepancy_overview(unadjusted_discrepancies, "variable")
unadjusted_discrepancy_summary

,variable,failed_rows,max_abs_difference,max_relative_difference


## 6. Validation 2: 04b Multivariable Model

**Python source**: `notebooks/04b_multivariable_outcomes.ipynb`  
**SAS source**: `sas/programs/04_multivariable_logistic.sas`  
**Files compared**: `python/outputs/python_logistic_parameters.csv` vs `sas/outputs/sas_logistic_parameters.csv`

Variables compared when exported by both platforms:

- `coefficient`
- `odds_ratio`
- `ci_lower`
- `ci_upper`
- `p_value`

`std_error` is not reconstructed in this notebook if a platform does not export it. This preserves the notebook's role as an artifact comparison layer rather than a model-output derivation layer.

Tolerances:

- Coefficients and p-values: absolute difference <= `1e-4`
- Odds ratios and confidence limits: absolute difference <= `1e-4`


In [7]:
LOGISTIC_COLUMN_RENAMES = {
    "odds_ratio_ci_lower": "ci_lower",
    "odds_ratio_ci_upper": "ci_upper",
}

TERM_ALIASES = {
    "anchor_year_group_2011 - 2013": "anchor_year_group_2011_2013",
    "anchor_year_group_2014 - 2016": "anchor_year_group_2014_2016",
    "anchor_year_group_2017 - 2019": "anchor_year_group_2017_2019",
    "anchor_year_group_2020 - 2022": "anchor_year_group_2020_2022",
}

LOGISTIC_KEY = ["term"]
LOGISTIC_REQUESTED_COLUMNS = [
    "coefficient",
    "std_error",
    "odds_ratio",
    "ci_lower",
    "ci_upper",
    "p_value",
]
LOGISTIC_NUMERIC_COLUMNS = LOGISTIC_REQUESTED_COLUMNS.copy()
LOGISTIC_ABS_TOLERANCES = {
    "coefficient": 1e-4,
    "std_error": 1e-4,
    "odds_ratio": 1e-4,
    "ci_lower": 1e-4,
    "ci_upper": 1e-4,
    "p_value": 1e-4,
}


def prepare_logistic(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    out = df.rename(columns=LOGISTIC_COLUMN_RENAMES).copy()
    out = out.loc[:, ~out.columns.duplicated()].copy()
    if "term" not in out.columns:
        raise ValueError(f"{platform} logistic output is missing column: term")

    available_columns = [col for col in LOGISTIC_REQUESTED_COLUMNS if col in out.columns]
    out = out[["term"] + available_columns].copy()
    out["term"] = out["term"].replace(TERM_ALIASES).astype(str)
    for col in available_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.sort_values(LOGISTIC_KEY).reset_index(drop=True)

python_logistic = prepare_logistic(python_logistic_raw, "Python")
sas_logistic = None if sas_logistic_raw is None else prepare_logistic(sas_logistic_raw, "SAS")

logistic_compare_columns = [
    col for col in LOGISTIC_REQUESTED_COLUMNS
    if col in python_logistic.columns and sas_logistic is not None and col in sas_logistic.columns
]

missing_from_python = [col for col in LOGISTIC_REQUESTED_COLUMNS if col not in python_logistic.columns]
missing_from_sas = [] if sas_logistic is None else [col for col in LOGISTIC_REQUESTED_COLUMNS if col not in sas_logistic.columns]

pd.DataFrame(
    {
        "metric": LOGISTIC_REQUESTED_COLUMNS,
        "compared": [col in logistic_compare_columns for col in LOGISTIC_REQUESTED_COLUMNS],
        "available_in_python": [col in python_logistic.columns for col in LOGISTIC_REQUESTED_COLUMNS],
        "available_in_sas": [False if sas_logistic is None else col in sas_logistic.columns for col in LOGISTIC_REQUESTED_COLUMNS],
    }
)


,metric,compared,available_in_python,available_in_sas
0,coefficient,True,True,True
1,std_error,False,False,True
2,odds_ratio,True,True,True
3,ci_lower,True,True,True
4,ci_upper,True,True,True
5,p_value,True,True,True


In [8]:
if sas_logistic is None:
    display(Markdown("TODO: SAS logistic output is unavailable; comparison table was not generated."))
    logistic_comparison = pd.DataFrame()
else:
    logistic_rows = []
    for variable in logistic_compare_columns:
        metric_comparison = compare_table(
            python_df=python_logistic[["term", variable]],
            sas_df=sas_logistic[["term", variable]],
            key_columns=LOGISTIC_KEY,
            compare_columns=[variable],
            numeric_columns=[variable],
            abs_tolerance=LOGISTIC_ABS_TOLERANCES[variable],
        )
        logistic_rows.append(metric_comparison)
    logistic_comparison = pd.concat(logistic_rows, ignore_index=True) if logistic_rows else pd.DataFrame()

comparison_overview("04b multivariable model", logistic_comparison)

,stage,comparison_rows,passed_rows,failed_rows,max_abs_difference
0,04b multivariable model,125,90,35,5.781118e+08


In [9]:
logistic_discrepancies = logistic_comparison.loc[
    logistic_comparison["pass"].eq(False)
].copy() if not logistic_comparison.empty else pd.DataFrame()

logistic_discrepancy_summary = discrepancy_overview(logistic_discrepancies, "variable")
logistic_discrepancy_summary

,variable,failed_rows,max_abs_difference,max_relative_difference
0,ci_upper,9,inf,NaN
1,coefficient,9,6.373303e+00,0.380964
2,p_value,9,1.772538e-01,0.179057
3,odds_ratio,8,5.781118e+08,0.998293


## 7. Notes

The full comparison tables are retained in memory as `unadjusted_comparison` and `logistic_comparison` for reproducibility. The displayed notebook output is intentionally compact: summary counts, grouped discrepancy diagnostics, and interpretation-level checks rather than repeated row-level tables.

Flagged differences should be reviewed in context. Differences can arise from categorical reference levels, dummy-variable coding, numeric precision, rounding, convergence behavior, or quasi-complete separation in sparse categories.


In [10]:
summary = pd.DataFrame(
    [
        {
            "stage": "04a unadjusted outcomes",
            "comparison_rows": len(unadjusted_comparison),
            "failed_rows": len(unadjusted_discrepancies),
        },
        {
            "stage": "04b multivariable model",
            "comparison_rows": len(logistic_comparison),
            "failed_rows": len(logistic_discrepancies),
        },
    ]
)
summary


,stage,comparison_rows,failed_rows
0,04a unadjusted outcomes,14,0
1,04b multivariable model,125,35


**README Figure Export**

The validation summary figure is saved under `assets/` for the portfolio README. It summarizes pass/fail counts from the comparison tables and does not change validation logic.


In [11]:
# Save README figure: SAS-Python validation summary
import matplotlib.pyplot as plt

assets_dir = PROJECT_ROOT / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)
figure_path = assets_dir / "sas_python_validation_summary.png"

plot_df = summary.copy()
plot_df["passed_rows"] = plot_df["comparison_rows"] - plot_df["failed_rows"]

fig, ax = plt.subplots(figsize=(8, 4.6))
y = np.arange(len(plot_df))
ax.barh(y, plot_df["passed_rows"], color="#059669", label="Passed")
ax.barh(y, plot_df["failed_rows"], left=plot_df["passed_rows"], color="#ea580c", label="Flagged")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["stage"])
ax.set_xlabel("Compared fields")
ax.set_title("SAS-Python aggregate output validation")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right")

for i, row in plot_df.iterrows():
    ax.text(
        row["comparison_rows"],
        i,
        f' {int(row["failed_rows"])} flagged / {int(row["comparison_rows"])} compared',
        va="center",
        fontsize=10,
        color="#0f172a",
    )

fig.text(
    0.01,
    0.01,
    "Validation compares aggregate outputs only; it is not a new clinical analysis.",
    fontsize=9,
    color="#475569",
)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

pd.DataFrame([{"exported_figure": str(figure_path.relative_to(PROJECT_ROOT))}])

,exported_figure
0,assets/sas_python_validation_summary.png


## 8. Focused Diagnostic Review: 04b Failed Rows

This section diagnoses the failed rows from the 04b SAS-Python model comparison. It does not change the Python or SAS output files and does not refit any model.

The diagnostic goal is to classify the failed comparison cells and determine whether they indicate a true model-specification problem, a validation-notebook issue, or an expected implementation difference between statsmodels and SAS PROC LOGISTIC.


In [12]:
MODEL_SPECIFICATION_CHECK = pd.DataFrame(
    [
        ("Outcome variable", "hospital_expire_flag", "hospital_expire_flag", "match"),
        ("Event definition", "event = 1", "event = '1'", "match"),
        ("Main exposure", "exposure = raas_any_early", "exposure = raas_any_early", "match"),
        ("Continuous covariate", "age", "age", "match"),
        ("Categorical covariates", "gender, race_group, admission_type, insurance, anchor_year_group", "gender_model, race_group_model, admission_type_model, insurance_model, anchor_year_group_model", "match after SAS display-variable mapping"),
        ("Reference coding", "pandas.get_dummies(..., drop_first=True)", "CLASS ... / PARAM=REF", "match by configured reference levels"),
        ("Intercept", "sm.add_constant(...)", "PROC LOGISTIC default intercept", "match"),
        ("Missing categorical values", "fillna('Unknown')", "missing() -> 'Unknown'", "match"),
        ("Model rows", "460,786", "expected 460,786 from nonicu_analysis", "match based on exported outcome count"),
        ("Events / non-events", "2,326 / 458,460", "2,326 / 458,460", "match based on exported unadjusted summaries"),
    ],
    columns=["check", "python_definition", "sas_definition", "assessment"],
)
MODEL_SPECIFICATION_CHECK[["check", "assessment"]]

,check,assessment
0,Outcome variable,match
1,Event definition,match
2,Main exposure,match
3,Continuous covariate,match
4,Categorical covariates,match after SAS display-variable mapping
5,Reference coding,match by configured reference levels
6,Intercept,match
7,Missing categorical values,match
8,Model rows,match based on exported outcome count
9,Events / non-events,match based on exported unadjusted summaries


In [13]:
if logistic_comparison.empty:
    failed_row_diagnostics = pd.DataFrame()
else:
    failed_row_diagnostics = logistic_comparison.loc[
        logistic_comparison["pass"].eq(False)
    ].copy()
    failed_row_diagnostics["term"] = failed_row_diagnostics["key"].str.replace("term=", "", regex=False)
    failed_row_diagnostics = failed_row_diagnostics.rename(
        columns={
            "variable": "statistic",
            "pass": "passed",
        }
    )
    failed_row_diagnostics["tolerance_used"] = failed_row_diagnostics["statistic"].map(LOGISTIC_ABS_TOLERANCES)
    failed_row_diagnostics["pass_fail_reason"] = "absolute_difference > tolerance_used"

    def classify_discrepancy(row):
        term = row["term"]
        statistic = row["statistic"]
        if term == "const" or term.startswith("admission_type_"):
            if statistic == "ci_upper":
                return "confidence interval mismatch"
            if statistic == "odds_ratio":
                return "coefficient estimate mismatch"
            return "convergence / optimizer difference"
        return "comparison-notebook issue"

    def recommend_action(row):
        term = row["term"]
        if term == "const" or term.startswith("admission_type_"):
            return "no action needed; document as expected implementation difference"
        return "fix validation merge/alias logic"

    failed_row_diagnostics["discrepancy_category"] = failed_row_diagnostics.apply(classify_discrepancy, axis=1)
    failed_row_diagnostics["recommended_action"] = failed_row_diagnostics.apply(recommend_action, axis=1)
    failed_row_diagnostics = failed_row_diagnostics[
        [
            "term",
            "statistic",
            "python_value",
            "sas_value",
            "absolute_difference",
            "relative_difference",
            "tolerance_used",
            "pass_fail_reason",
            "discrepancy_category",
            "recommended_action",
        ]
    ]

if failed_row_diagnostics.empty:
    failed_diagnostic_counts = pd.DataFrame(columns=["discrepancy_category", "statistic", "failed_rows", "affected_term_count"])
else:
    failed_diagnostic_counts = (
        failed_row_diagnostics
        .groupby(["discrepancy_category", "statistic"], as_index=False)
        .agg(
            failed_rows=("term", "size"),
            affected_term_count=("term", "nunique"),
            max_abs_difference=("absolute_difference", "max"),
        )
        .sort_values(["failed_rows", "discrepancy_category", "statistic"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
failed_diagnostic_counts

,discrepancy_category,statistic,failed_rows,affected_term_count,max_abs_difference
0,confidence interval mismatch,ci_upper,9,9,inf
1,convergence / optimizer difference,coefficient,9,9,6.373303e+00
2,convergence / optimizer difference,p_value,9,9,1.772538e-01
3,coefficient estimate mismatch,odds_ratio,8,8,5.781118e+08


In [14]:
if failed_row_diagnostics.empty:
    diagnostic_summary = pd.DataFrame()
else:
    diagnostic_summary = (
        failed_row_diagnostics
        .groupby("discrepancy_category", as_index=False)
        .agg(
            failed_rows=("term", "size"),
            affected_term_count=("term", "nunique"),
            affected_statistics=("statistic", lambda values: ", ".join(sorted(set(values)))),
            term_examples=("term", lambda values: ", ".join(sorted(set(values))[:4])),
            recommended_action=("recommended_action", "first"),
        )
    )
    diagnostic_summary["likely_cause"] = diagnostic_summary["discrepancy_category"].map(
        {
            "convergence / optimizer difference": "Intercept and non-reference admission_type coefficients shift by the same amount in opposite directions, consistent with optimizer-specific finite estimates under sparse or quasi-separated admission_type strata.",
            "coefficient estimate mismatch": "Odds-ratio differences are downstream of the admission_type coefficient offset; exp(coefficient) is applied consistently in both outputs.",
            "confidence interval mismatch": "Python reports infinite upper confidence limits for the affected sparse terms, while SAS reports finite but extremely large Wald limits.",
            "comparison-notebook issue": "Unexpected failure outside the known intercept/admission_type pattern.",
        }
    )
    diagnostic_summary = diagnostic_summary[
        [
            "discrepancy_category",
            "failed_rows",
            "affected_term_count",
            "affected_statistics",
            "term_examples",
            "likely_cause",
            "recommended_action",
        ]
    ]

diagnostic_summary

,discrepancy_category,failed_rows,affected_term_count,affected_statistics,term_examples,likely_cause,recommended_action
0,coefficient estimate mismatch,8,8,odds_ratio,"admission_type_DIRECT EMER., admission_type_DI...",Odds-ratio differences are downstream of the a...,no action needed; document as expected impleme...
1,confidence interval mismatch,9,9,ci_upper,"admission_type_DIRECT EMER., admission_type_DI...",Python reports infinite upper confidence limit...,no action needed; document as expected impleme...
2,convergence / optimizer difference,18,9,"coefficient, p_value","admission_type_DIRECT EMER., admission_type_DI...",Intercept and non-reference admission_type coe...,no action needed; document as expected impleme...


In [15]:
# Diagnostic invariance check: for non-reference admission_type rows,
# the intercept + admission_type linear predictor is nearly identical across platforms.
if sas_logistic is None:
    admission_type_linear_predictor_check = pd.DataFrame()
else:
    py_terms = python_logistic.set_index("term")
    sas_terms = sas_logistic.set_index("term")
    rows = []
    for term in sorted([t for t in py_terms.index if t.startswith("admission_type_")]):
        if term in sas_terms.index:
            python_value = py_terms.loc["const", "coefficient"] + py_terms.loc[term, "coefficient"]
            sas_value = sas_terms.loc["const", "coefficient"] + sas_terms.loc[term, "coefficient"]
            rows.append(
                {
                    "term": term,
                    "python_intercept_plus_term": python_value,
                    "sas_intercept_plus_term": sas_value,
                    "absolute_difference": abs(sas_value - python_value),
                }
            )
    admission_type_linear_predictor_check = pd.DataFrame(rows)

if admission_type_linear_predictor_check.empty:
    admission_type_linear_predictor_summary = pd.DataFrame()
else:
    admission_type_linear_predictor_summary = pd.DataFrame(
        [
            {
                "checked_terms": len(admission_type_linear_predictor_check),
                "max_abs_difference": admission_type_linear_predictor_check["absolute_difference"].max(),
                "median_abs_difference": admission_type_linear_predictor_check["absolute_difference"].median(),
            }
        ]
    )

admission_type_linear_predictor_summary

,checked_terms,max_abs_difference,median_abs_difference
0,8,4.211854e-09,3.668966e-09


### 8.1 Diagnostic Interpretation

The failed rows are limited to the intercept and non-reference `admission_type` terms. Exposure, age, gender, race group, insurance, and anchor-year terms match within tolerance.

The coefficient pattern is structured: SAS estimates for the intercept are higher than Python by the same amount that SAS estimates for each non-reference `admission_type` coefficient are lower than Python. The combined intercept plus `admission_type` linear predictors are essentially identical across platforms. This indicates that the fitted risks for non-reference admission-type groups are aligned, while the finite split between the intercept and sparse admission-type contrasts differs by optimizer implementation.

No failed row suggests a coefficient-vs-odds-ratio comparison bug. Odds ratios equal `exp(coefficient)` in both outputs, and confidence interval differences are downstream of the same sparse-category / convergence behavior.

---
## Next Step

Proceed to:
- [Repository README](../README.md)

This next notebook/document briefly describes the next workflow step.
